# Import required libraries

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import math
from sklearn.model_selection import TimeSeriesSplit, StratifiedGroupKFold, cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, make_scorer, fbeta_score, precision_score, recall_score
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# Only select the years 2019, 2022, 2023, 2024, and 2025 
files_and_ranges = [
    ("I:/Datasets Thesis/1 Original Weather Dataset/uurgeg_260_2011-2020.txt", "2019-01-01", "2019-12-31"),
    ("I:/Datasets Thesis/1 Original Weather Dataset/uurgeg_260_2021-2030.txt", "2022-01-01", "2025-12-31"),
]

weather = pd.concat(
    [
        pd.read_csv(
            path,
            sep=",",
            skiprows=30,
            parse_dates=["YYYYMMDD"],
            low_memory=False
        ).loc[lambda d: d["YYYYMMDD"].between(start, end)]
        for path, start, end in files_and_ranges
    ],
    ignore_index=True
)

In [3]:
weather.columns = weather.columns.str.strip().str.lstrip("#").str.strip()

weather = (
    weather.rename(columns={
        "STN": "Station",
        "YYYYMMDD": "Date",
        "HH": "Hour",
        "DD": "Wind Direction",
        "FH": "Hourly Mean Wind Speed",
        "FF": "Wind Speed last 10 Minutes",
        "FX": "Max Wind Speed",
        "T": "Temperature",
        "T10N": "Min Temperature",
        "TD": "Dew Point temperature",
        "SQ": "Sunshine Duration",
        "Q": "Global Radiation",
        "DR": "Precipitation Duration",
        "RH": "Precipitation Amount",
        "P": "Air Pressure",
        "VV": "Horizontal Visibility",
        "N": "Cloud Cover",
        "U": "Humidity",
        "WW": "Weather Code",
        "IX": "Indicator weather code",
        "M": "Fog",
        "R": "Rainfall",
        "S": "Snowfall",
        "O": "Thunder",
        "Y": "Hail",
    })
)

num_cols = weather.columns.drop("Date")
weather[num_cols] = weather[num_cols].apply(pd.to_numeric, errors="coerce", downcast="float")

In [4]:
s = weather["Cloud Cover"]
gaps = s.isna().astype(int)
run_id = (gaps != gaps.shift()).cumsum()
missing_runs = gaps.groupby(run_id).sum()
missing_runs = missing_runs[missing_runs > 0]

print(missing_runs.describe())
print(missing_runs.value_counts().sort_index())

count     2.000000
mean     22.500000
std      28.991378
min       2.000000
25%      12.250000
50%      22.500000
75%      32.750000
max      43.000000
Name: Cloud Cover, dtype: float64
Cloud Cover
2     1
43    1
Name: count, dtype: int64


In [5]:
weather.groupby(weather["Date"].dt.month)["Cloud Cover"].agg(["mean", "median", "std", "count"])

,mean,median,std,count
Date,,,,
1,6.754839,8.0,2.621319,3720
2,6.429374,8.0,2.870705,3384
3,5.498117,8.0,3.419999,3718
4,5.588611,8.0,3.298005,3600
5,5.688710,8.0,3.161577,3720
6,4.976111,7.0,3.434779,3600
7,5.854839,8.0,2.982138,3720
8,5.340054,7.0,3.263495,3720
9,5.885859,8.0,3.042151,3557


In [6]:
before = weather["Cloud Cover"].value_counts(normalize=True).sort_index()
after = weather["Cloud Cover"].ffill().value_counts(normalize=True).sort_index()

comparison = pd.concat([before, after], axis=1)
comparison.columns = ["before_ffill", "after_ffill"]
comparison["difference"] = comparison["after_ffill"] - comparison["before_ffill"]
print(comparison)
print("Max absolute distribution change:", comparison["difference"].abs().max())

             before_ffill  after_ffill  difference
Cloud Cover                                       
0.0              0.136687     0.136546   -0.000140
1.0              0.043742     0.043698   -0.000045
2.0              0.024578     0.024553   -0.000025
3.0              0.021746     0.021723   -0.000022
4.0              0.023687     0.023663   -0.000024
5.0              0.023413     0.023389   -0.000024
6.0              0.030540     0.030508   -0.000031
7.0              0.082620     0.083516    0.000896
8.0              0.602846     0.602273   -0.000573
9.0              0.010142     0.010131   -0.000010
Max absolute distribution change: 0.0008963609338520917
